# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired.

As we use defaults for *E.coli* for most of the sector parameters, the determination of the slope and intercept for the sector parameters can be found at the end of this notebook. 

In [1]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime
import math

from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')

from Modules.utils.pamparametrizer_setup import save_sector_information_to_excel
from Modules.utils.preprocessing import *

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iJN1463_EnzymaticData_basis.xlsx')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_pputidakt2240_240807.xlsx')

GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_ppu_240807.json')

Loading PAModelpy modules version 0.0.5.1
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-03


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iJN1463_EnzymaticData_{formatted_date}.xlsx')

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BiGG database](http://bigg.ucsd.edu/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iJN1463 model for [*Pseudomonas putida* KT2240](http://bigg.ucsd.edu/models/iJN1463)
- Important: the identifiers must me mappable to uniprot (or another enzyme identifier database) and KEGG (for mapping to GotEnzymes)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `ppu` (*P. putida* strain KT2240) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be generated
- Once the webpage is finished processing your query, scroll down and copy/download the resulting json file
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *P. putida* we used [this](https://www.uniprot.org/uniprotkb?query=%22pseudomonas+putida%22+AND+%28organism_id%3A160488%29) (strain: *Pseudomonas putida (strain ATCC 47054 / DSM 6125 / CFBP 8728 / NCIMB 11950 / KT2440)*) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Mark `Compressed` as `No`
- Download the resulting Excel file

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [3]:
#load the model
model = read_sbml_model(os.path.join('Models', 'iJN1463.xml'))
id_mapper_df = create_id_mapper_from_model(model).rename({'ec-code': 'EC'}, axis=1)

Mapped 701 of 2181 reactions to KEGG reaction IDs.


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 2.1 Parse the BIGG gene-protein-reaction associations

In [4]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'\b(PP_\d{4})\b'
def extract_locustags(text):
    return re.findall(locustag_regex, text)

id_mapper_df['locus_tag'] = id_mapper_df['GPR'].apply(extract_locustags)
id_mapper_df = id_mapper_df.explode('locus_tag', ignore_index=True)
id_mapper_df

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag
0,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,"[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602
1,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,"[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_4174
2,13DAMPPabcpp,"[nan, C00002, [C00001, C01328]]","[nan, C00008, C00080, C00009]",,False,NaN,NaN,NaN
3,15DAPabcpp,"[C01672, C00002, [C00001, C01328]]","[C01672, C00008, C00080, C00009]",,False,NaN,NaN,NaN
4,1P2CBXLCYCL,[C01110],"[nan, [C00001, C01328], C00080]",PP_s0001,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
3991,4HTHRS,"[[C00001, C01328], C06055]","[C06056, C00009]",PP_1471 or PP_0662,False,R05086,"[4.2.3.1, 4.2.3]",PP_0662
3992,4MBZALDH,"[C06757, C00003]","[C06758, C00080, C00004]",pWW0_128,False,R05282,"[1.1.1.90, 1.1.1.21]",NaN
3993,4MBZDH,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]",pWW0_131,False,NaN,"[1.2.1.3, 1.2.1.28, 1.2.1.7]",NaN
3994,4MCAT23DOX,"[C06730, C00007]","[C00080, C06760]",pWW0_097,False,R05295,1.13.11.2,NaN


### 2.2 Parse the Uniprot data for merging

In [5]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['locus_tag'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
uniprot_df

/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Entry,Protein names,Length,Mass,Rhea ID,locus_tag
0,Q88CC1,2-oxoadipate dioxygenase/decarboxylase (EC 1.1...,464,51372,RHEA:71787,PP_5260
1,Q88E10,Methyl-accepting chemotaxis protein McpS,639,68764,NaN,PP_4658
2,Q88E47,"Homogentisate 1,2-dioxygenase (HGDO) (EC 1.13....",433,48011,RHEA:15449,PP_4621
3,Q88F88,Pyoverdine export ATP-binding/permease protein...,654,69687,NaN,PP_4210
4,Q88FF8,Quinone reductase (EC 1.6.5.2) (Chromate reduc...,186,20354,RHEA:46160 RHEA:46164 RHEA:44372 RHEA:44368,PP_4138
...,...,...,...,...,...,...
5524,Q88RW2,Transcriptional regulator,119,13492,NaN,PP_0017
5525,Q88RW3,TniQ family protein,638,71170,NaN,PP_0016
5526,Q88RW4,Transposon protein,319,36304,NaN,PP_0015
5527,Q88RW5,Transposase,652,74345,NaN,PP_0014


### 2.3 Match Uniprot to BIGG data

In [6]:
id_mapper_with_protein = pd.merge(id_mapper_df, 
                                  uniprot_df, on='locus_tag', 
                                  how='left').drop(['Rhea ID', 'Protein names'], 
                                                   axis=1)

id_mapper_with_protein

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,Entry,Length,Mass
0,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,"[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602,Q88MG9,146.0,16583.0
1,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,"[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_4174,Q88FC4,171.0,18791.0
2,13DAMPPabcpp,"[nan, C00002, [C00001, C01328]]","[nan, C00008, C00080, C00009]",,False,NaN,NaN,NaN,G1JLS8,205.0,21908.0
3,13DAMPPabcpp,"[nan, C00002, [C00001, C01328]]","[nan, C00008, C00080, C00009]",,False,NaN,NaN,NaN,V6C0L5,14.0,1529.0
4,15DAPabcpp,"[C01672, C00002, [C00001, C01328]]","[C01672, C00008, C00080, C00009]",,False,NaN,NaN,NaN,G1JLS8,205.0,21908.0
...,...,...,...,...,...,...,...,...,...,...,...
4250,4MBZDH,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]",pWW0_131,False,NaN,"[1.2.1.3, 1.2.1.28, 1.2.1.7]",NaN,V6C0L5,14.0,1529.0
4251,4MCAT23DOX,"[C06730, C00007]","[C00080, C06760]",pWW0_097,False,R05295,1.13.11.2,NaN,G1JLS8,205.0,21908.0
4252,4MCAT23DOX,"[C06730, C00007]","[C00080, C06760]",pWW0_097,False,R05295,1.13.11.2,NaN,V6C0L5,14.0,1529.0
4253,4OD,"[C00080, C03453]","[C00011, nan]",pWW0_091,False,NaN,NaN,NaN,G1JLS8,205.0,21908.0


### 3. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [7]:
id_mapper_final = id_mapper_with_protein
id_mapper_final = id_mapper_final.rename({'Entry':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg.reaction', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,enzyme_id,Length,Mass
0,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,2.3.1.85,PP_1602,Q88MG9,146.0,16583.0
1,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,2.3.1.86,PP_1602,Q88MG9,146.0,16583.0
2,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,4.2.1.61,PP_1602,Q88MG9,146.0,16583.0
3,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,4.2.1.59,PP_1602,Q88MG9,146.0,16583.0
4,3HAD160,[C04633],"[[C00001, C01328], C05763]",PP_1602 or PP_4174,False,R04544,2.3.1.-,PP_1602,Q88MG9,146.0,16583.0


In [8]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,PP_0004,R08701,,C00101,3.6784
1,PP_0004,R08701,,C00037,1.4164
2,PP_0004,R08701,,C03479,3.2723
3,PP_0005,R08701,,C00101,2.4913
4,PP_0005,R08701,,C03479,2.1913


In [9]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = map_kcat_values_to_reaction_protein_association(id_mapper = id_mapper_final, 
                                                                     gotenzymes_df = eco_enzymes_df)
eco_enzymes_merged

Mapped 4695 out of 8676 kcat values to reactions based on kegg gene id
Mapped 4727 out of 8676 kcat values to reactions based on ec number
Final merged dataset has 5016 unique reactions with kcat values.


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,Length,Mass,gene,ec_number,compound,kcat_values
0,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1.5.1.1,C00408,86.6017
1,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1.5.1.21,C00408,86.6017
2,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1.5.1.1,C04092,1422.1744
3,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1.5.1.21,C04092,1422.1744
4,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.25,Q88GX6,332.0,35143.0,PP_3591,1.5.1.1,C00408,86.6017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8732,4MBZDH,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]",pWW0_131,False,NaN,1.2.1.7,V6C0L5,14.0,1529.0,NaN,NaN,NaN,NaN
8733,4MCAT23DOX,"[C06730, C00007]","[C00080, C06760]",pWW0_097,False,R05295,1.13.11.2,G1JLS8,205.0,21908.0,NaN,NaN,NaN,NaN
8734,4MCAT23DOX,"[C06730, C00007]","[C00080, C06760]",pWW0_097,False,R05295,1.13.11.2,V6C0L5,14.0,1529.0,NaN,NaN,NaN,NaN
8735,4OD,"[C00080, C03453]","[C00011, nan]",pWW0_091,False,NaN,NaN,G1JLS8,205.0,21908.0,NaN,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [10]:
# Get directionalities and fill in the gaps
eco_enzymes_parsed = assign_directionalities_for_kcat_relations(eco_enzymes_merged.copy())
eco_enzymes_parsed = assign_defaults_for_proteins_without_mapping(eco_enzymes_parsed)

# Handle specific cases
#PP_s0001 is a gene which is not mappable
eco_enzymes_parsed.loc[eco_enzymes_parsed['GPR'] == 'PP_s0001', ['gene', 'enzyme_id']] = eco_enzymes_parsed.apply(
    lambda row: ('PP_s0001', f'Enzyme_s0001_{row.rxn_id}'),
    axis=1, result_type="expand"
)
#pWW0 is a suffix included in the gene ids for genes on a plasmid
eco_enzymes_parsed.loc[
    eco_enzymes_parsed['GPR'].str.contains('pWW0', na=False), 
    'gene'
] = eco_enzymes_parsed['GPR']

eco_enzymes_parsed

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,Length,molMass,gene,kcat_values,direction
0,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,86.6017,b
1,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,86.6017,b
2,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1422.1744,f
3,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.1,Q88GX6,332.0,35143.0,PP_3591,1422.1744,f
4,1PPDCRc,"[C04092, C00080, C00005]","[C00408, C00006]",PP_3591,False,R02203,1.5.1.25,Q88GX6,332.0,35143.0,PP_3591,86.6017,b
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9285,VCOAD,"[C00016, C00888]","[C01352, C02451]",PP_1893 or PP_2216,True,NaN,NaN,Q88LN6,815.0,89195.0,PP_1893,13.7000,b
9286,VCOAD,"[C00016, C00888]","[C01352, C02451]",PP_1893 or PP_2216,True,NaN,NaN,Q88KS3,375.0,40529.0,PP_2216,13.7000,b
9287,VECOAH,[nan],"[[C00001, C01328], C02451]",PP_2136 or PP_2217,True,NaN,NaN,Q88L02,715.0,77450.0,PP_2136,13.7000,b
9288,VECOAH,[nan],"[[C00001, C01328], C02451]",PP_2136 or PP_2217,True,NaN,NaN,Q88KS2,257.0,27668.0,PP_2217,13.7000,b


In [11]:
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_parsed, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_final = merge_enzyme_complexes(eco_enzymes_parsed, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_final.drop_duplicates(subset = ['rxn_id', 'enzyme_id', 'direction'], inplace = True)

### 7. Save the dataframe to the proper format for building PAMs

In [12]:
other_sheets = pd.read_excel(PAM_DATA_FILE_PATH, sheet_name = None)
del other_sheets['ActiveEnzymes']
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes', index=False)
    for sheet_name, sheet in other_sheets.items():
        if sheet_name == 'ExcessEnzymes': sheet_name = 'UnusedEnzyme'
        sheet.to_excel(writer, sheet_name = sheet_name, index=False)

## 8. Store the information about the unused enzyme sector
For the translational sector we use the growth-rate to translational protein relation from E.coli, which will automatically be calculated if it is not supplied.

In [13]:
MAX_GROWTH_ALE_ECOLI = math.log(2)*1.63#Or from lenski experiment which is ln(2)*1.63(doublings per hour)
MAX_GROWTH_RATE_ECOLI = 0.67
MEASURED_PROTEIN_FRACTION = 0.55*0.55
fraction_growth_rate_increase = MAX_GROWTH_ALE_ECOLI/MAX_GROWTH_RATE_ECOLI

BIOMASS_RXN_ID = 'BIOMASS_KT2440_WT3'
MAX_GROWTH_RATE_PUTIDA = 0.73 #10.1128/AEM.05588-11

ups_0  = 0.37*MEASURED_PROTEIN_FRACTION 
ups_mu = -ups_0/(MAX_GROWTH_RATE_PUTIDA*fraction_growth_rate_increase)

#dummy data for relation to substrate_uptake rate
save_sector_information_to_excel(
        param_vs_lin_rxn={
            'slope': -ups_mu,
            'intercept': ups_0
        },
        param_vs_growth={
            'slope': ups_mu,
            'intercept': ups_0
        },
        biomass_rxn=BIOMASS_RXN_ID,
        lin_rxn_id='EX_glc__D_e',
        sector_id='UnusedEnzyme',
        pam_data_file=OUTPUT_FILE_PATH, model_name = 'iJN1463')

Results/1_preprocessing/proteinAllocationModel_iJN1463_EnzymaticData_250915.xlsx
         Parameter         Value    Value_for_growth        Unit  \
0          id_list   EX_glc__D_e  BIOMASS_KT2440_WT3         NaN   
1           ups_mu      0.090921           -0.090921   g/g_CDW/h   
2            ups_0      0.111925            0.111925     g/g_CDW   
3         mol_mass  39477.787843        39477.787843       g/mol   
4  substrate_range           NaN    [-4, -3, -2, -1]  mmol/gDW/h   

                                         Description  
0  biomass formation reaction ID (representing gr...  
1  slope of relation between growth rate and prot...  
2  Protein concentration allocated to excess enzy...  
3                   Molar mass of a fictional enzyme  
4  Range of values of susbstrate uptake which are...  
